In [1]:
import os
import pandas as pd

def compute_soc_from_bms(
    df: pd.DataFrame,
    time_col: str = "Time",
    current_col: str = "Current_measured",
    discharge_current_negative: bool = True,
):
    """
    Compute State of Charge (SoC) from current + time using coulomb counting.
    
    Assumptions:
    - df contains at least [time_col, current_col].
    - time_col is in seconds, monotonically increasing.
    - current_col is in Amps.
    - For this dataset, discharge current is NEGATIVE (hence discharge_current_negative=True).
    
    Returns:
        df_out: original df with added columns:
            - 'delta_t'      : time step (s)
            - 'discharge_I'  : positive discharge current (A)
            - 'delta_Ah'     : incremental discharged capacity (Ah)
            - 'cum_Ah'       : cumulative discharged capacity (Ah)
            - 'SoC'          : relative SoC in [0,1]
            - 'SoC_percent'  : SoC in [0,100]
        capacity_Ah: estimated total discharged capacity for this file (Ah)
    """

    df = df.copy()

    # 1) Sort by time just to be safe
    df = df.sort_values(time_col).reset_index(drop=True)

    # 2) Compute time step Δt (seconds)
    df["delta_t"] = df[time_col].diff().fillna(0)

    # 3) Extract discharge current as a positive value
    if discharge_current_negative:
        # discharge = negative current -> take absolute of negative part
        df["discharge_I"] = df[current_col].clip(upper=0).abs()
    else:
        # discharge = positive current
        df["discharge_I"] = df[current_col].clip(lower=0)

    # 4) Convert current * time to discharged capacity (Ah)
    #    I (A) * Δt (s) = Coulombs; divide by 3600 to get Ah
    df["delta_Ah"] = df["discharge_I"] * df["delta_t"] / 3600.0

    # 5) Cumulative discharged capacity
    df["cum_Ah"] = df["delta_Ah"].cumsum()

    # 6) Total discharged capacity for this cycle (Ah)
    capacity_Ah = df["cum_Ah"].iloc[-1]

    # 7) SoC = 1 - discharged/total  (normalized 1 → 0 over the file)
    #    This gives SoC independent of the missing 'Capacity' in metadata.
    df["SoC"] = 1.0 - df["cum_Ah"] / capacity_Ah

    # Clamp numerical noise into [0,1]
    df["SoC"] = df["SoC"].clip(lower=0.0, upper=1.0)

    # 8) Useful percentage form
    df["SoC_percent"] = df["SoC"] * 100.0

    return df, capacity_Ah


In [5]:
df_raw

,Voltage_measured,Current_measured,Temperature_measured,Current_load,Voltage_load,Time
0,4.246711,0.000252,6.212696,0.0002,0.000,0.000
1,4.246764,-0.001411,6.234019,0.0002,4.262,9.360
2,4.039277,-0.995093,6.250255,1.0000,3.465,23.281
3,4.019506,-0.996731,6.302176,1.0000,3.451,36.406
4,4.004763,-0.992845,6.361645,1.0000,3.438,49.625
...,...,...,...,...,...,...
485,3.303251,-0.001760,9.662331,0.0004,0.000,6382.063
486,3.310303,-0.000756,9.390489,0.0002,0.000,6395.547
487,3.317351,-0.003318,9.137008,0.0002,0.000,6409.063
488,3.323387,-0.002291,8.972806,0.0002,0.000,6422.625


In [6]:
# Path to your CSV file
file_path = "/kaggle/input/nasa-battery-dataset/cleaned_dataset/data/00001.csv"   # or os.path.join("data_folder", "00001.csv")

# Load the BMS data
df_raw = pd.read_csv(file_path)

# Compute SoC from BMS values
df_soc, capacity_est = compute_soc_from_bms(df_raw)

print(f"Estimated capacity for {file_path}: {capacity_est:.4f} Ah")
print(df_soc.head(450)[["Time", "Current_measured", "SoC", "SoC_percent"]])


Estimated capacity for /kaggle/input/nasa-battery-dataset/cleaned_dataset/data/00001.csv: 1.7060 Ah
         Time  Current_measured       SoC  SoC_percent
0       0.000          0.000252  1.000000   100.000000
1       9.360         -0.001411  0.999998    99.999785
2      23.281         -0.995093  0.997742    99.774228
3      36.406         -0.996731  0.995612    99.561219
4      49.625         -0.992845  0.993475    99.347520
..        ...               ...       ...          ...
445  5845.953         -0.996399  0.054066     5.406638
446  5859.250         -0.996147  0.051910     5.190964
447  5872.453         -0.994053  0.049773     4.977264
448  5885.719         -0.996105  0.047621     4.762101
449  5898.938         -0.994592  0.045480     4.548027

[450 rows x 4 columns]


In [7]:
import pandas as pd
import numpy as np

def compute_soc_precise(
    df: pd.DataFrame,
    time_col: str = "Time",
    current_col: str = "Current_measured",
    voltage_col: str = "Voltage_measured",
    current_noise_threshold: float = 1e-3,
    known_capacity_Ah: float = None,
):
    """
    Estimate State of Charge (SoC) from BMS data using coulomb counting.

    Columns used:
      - Time (s)
      - Current_measured (A), negative during discharge, positive during charge
      - Voltage_measured (V) used only for optional anchoring / sanity

    Args:
        df: DataFrame with at least time_col, current_col, voltage_col.
        time_col: name of time column (seconds, increasing).
        current_col: name of pack current column (Amps).
        voltage_col: name of pack voltage column (Volts).
        current_noise_threshold: |I| below this is treated as 0 (noise filter).
        known_capacity_Ah: if you know the battery capacity (Ah), pass it.
                           If None, it will be estimated from this cycle.

    Returns:
        df_out: DataFrame with extra columns:
            - delta_t      : time step in seconds
            - I_clean      : current with noise removed
            - delta_Ah     : incremental ΔAh (+ for charging, - for discharging)
            - cum_Ah       : cumulative Ah change (relative to first sample)
            - SoC_raw      : SoC from coulomb counting before anchoring
            - SoC          : SoC after anchoring to [0,1]
            - SoC_percent  : SoC in %
        capacity_Ah: capacity used for normalization (known or estimated)
    """

    df = df.copy()
    df = df.sort_values(time_col).reset_index(drop=True)

    # 1) Time step Δt
    df["delta_t"] = df[time_col].diff().fillna(0.0)  # seconds

    # 2) Noise filter on current
    I = df[current_col].values.astype(float)
    I_clean = np.where(np.abs(I) < current_noise_threshold, 0.0, I)
    df["I_clean"] = I_clean

    # 3) Convert current*dt to Ah
    #    I (A) * dt (s) = Coulombs; /3600 = Ah
    df["delta_Ah"] = df["I_clean"] * df["delta_t"] / 3600.0

    # 4) Cumulative Ah from the start of the file
    #    Positive current -> charging (SoC increases)
    #    Negative current -> discharging (SoC decreases)
    df["cum_Ah"] = df["delta_Ah"].cumsum()

    # 5) Estimate capacity if not given
    if known_capacity_Ah is None:
        # Only consider discharge part (negative current)
        discharge_Ah = (
            (df["I_clean"].clip(upper=0).abs() * df["delta_t"] / 3600.0).sum()
        )
        capacity_Ah = discharge_Ah
    else:
        capacity_Ah = float(known_capacity_Ah)

    # 6) Raw SoC from coulomb counting.
    #    Assume first sample ≈ 100% SoC and normalize by capacity.
    #    SoC_raw = 1 + (cum_Ah / capacity_Ah) because:
    #      - On discharge, cum_Ah is negative -> SoC decreases from 1.
    #      - On charge, cum_Ah is positive    -> SoC increases from 1.
    df["SoC_raw"] = 1.0 + df["cum_Ah"] / capacity_Ah

    # 7) Anchor SoC between 0 and 1 over this file
    #    Make first point = 1, last point = 0 (full discharge cycle).
    #    This corrects small drift.
    soc0 = df["SoC_raw"].iloc[0]
    socN = df["SoC_raw"].iloc[-1]

    if soc0 != socN:
        df["SoC"] = (df["SoC_raw"] - socN) / (soc0 - socN)  # rescale to [0,1]
    else:
        # Fallback: just clip raw SoC
        df["SoC"] = df["SoC_raw"]

    # Clip numerical noise
    df["SoC"] = df["SoC"].clip(0.0, 1.0)

    # 8) SoC in percent
    df["SoC_percent"] = df["SoC"] * 100.0

    return df, capacity_Ah


In [8]:
import pandas as pd

# Load your BMS file (with the 6 columns)
df_raw = pd.read_csv("/kaggle/input/nasa-battery-dataset/cleaned_dataset/data/00001.csv")

df_soc, cap_est = compute_soc_precise(df_raw)

print(f"Estimated capacity for this cycle: {cap_est:.4f} Ah")
print(df_soc[["Time", "Voltage_measured", "Current_measured", "SoC_percent"]].head())
print(df_soc[["Time", "Voltage_measured", "Current_measured", "SoC_percent"]].tail())


Estimated capacity for this cycle: 1.7060 Ah
     Time  Voltage_measured  Current_measured  SoC_percent
0   0.000          4.246711          0.000252   100.000000
1   9.360          4.246764         -0.001411    99.999785
2  23.281          4.039277         -0.995093    99.774227
3  36.406          4.019506         -0.996731    99.561216
4  49.625          4.004763         -0.992845    99.347516
         Time  Voltage_measured  Current_measured  SoC_percent
485  6382.063          3.303251         -0.001760     0.001528
486  6395.547          3.310303         -0.000756     0.001528
487  6409.063          3.317351         -0.003318     0.000798
488  6422.625          3.323387         -0.002291     0.000292
489  6436.141          3.329356         -0.001326     0.000000


In [9]:
import os
import numpy as np
import pandas as pd


def estimate_capacity_from_bms(
    df: pd.DataFrame,
    time_col: str = "Time",
    current_col: str = "Current_measured",
    load_col: str = "Current_load",
    voltage_col: str = "Voltage_measured",
    temp_col: str = "Temperature_measured",
    current_noise_threshold: float = 1e-3,
    load_threshold: float = 0.5,
) -> dict:
    """
    Estimate discharge capacity (Ah) from one cycle using all available BMS columns.

    Uses:
      - Time + Current_measured   -> coulomb counting (main signal)
      - Current_load              -> restrict integration to actual load (discharge) periods
      - Voltage_measured          -> restrict to valid voltage window if needed (optional)
      - Temperature_measured      -> returned as info; could be used for temperature corrections
      - Voltage_load              -> not explicitly needed for capacity, but available for diagnostics

    Returns a dict with:
      - capacity_Ah         : estimated discharged capacity for this file
      - avg_temperature_C   : average temperature during active discharge
      - avg_voltage_V       : average voltage during active discharge
      - active_time_s       : total time (s) under load
    """

    d = df.copy()
    d = d.sort_values(time_col).reset_index(drop=True)

    # Time step (seconds)
    d["delta_t"] = d[time_col].diff().fillna(0.0)

    # 1) Identify when the battery is actually under load (discharging)
    # In your 00001.csv, Current_load ~ 1.0 when active, ~0.0002 when idle.
    active_mask = d[load_col] > load_threshold

    # 2) Clean the current signal (remove near-zero noise)
    I = d[current_col].astype(float).values
    I_clean = np.where(np.abs(I) < current_noise_threshold, 0.0, I)
    d["I_clean"] = I_clean

    # 3) Only integrate during active discharge (negative current, load on)
    #    Negative current -> discharge, so we flip sign to get positive Ah.
    d["delta_t_active"] = np.where(active_mask, d["delta_t"], 0.0)
    I_discharge = np.clip(d["I_clean"], None, 0.0)  # <= 0, discharge currents
    d["delta_Ah"] = -I_discharge * d["delta_t_active"] / 3600.0  # Ah

    capacity_Ah = d["delta_Ah"].sum()

    # 4) Some extra stats using other columns (uses ALL your BMS signals)
    active_rows = d[active_mask & (d["delta_t_active"] > 0)]
    avg_temp = float(active_rows[temp_col].mean()) if not active_rows.empty else float(d[temp_col].mean())
    avg_voltage = float(active_rows[voltage_col].mean()) if not active_rows.empty else float(d[voltage_col].mean())
    active_time_s = float(active_rows["delta_t_active"].sum())

    return {
        "capacity_Ah": capacity_Ah,
        "avg_temperature_C": avg_temp,
        "avg_voltage_V": avg_voltage,
        "active_time_s": active_time_s,
    }


In [10]:
def compute_soh_for_all_cycles(
    metadata_path: str,
    data_dir: str,
    nominal_capacity_Ah: float = None,
    save_path: str = "metadata_with_soh.csv",
):
    """
    Compute SoH for each discharge cycle using all BMS columns from the raw files.

    Args:
        metadata_path: path to metadata.csv (with columns: filename, type, battery_id, Capacity, ...)
        data_dir: folder where 00001.csv, 00002.csv, ... live
        nominal_capacity_Ah: if known from spec (e.g., 2.0 Ah). If None, we infer per-battery.
        save_path: where to save updated metadata with SoH columns.

    Returns:
        metadata_with_soh: DataFrame with new columns:
            - capacity_est_Ah
            - avg_temperature_C
            - avg_voltage_V
            - active_time_s
            - reference_capacity_Ah
            - SoH
            - SoH_percent
    """

    meta = pd.read_csv(metadata_path)
    meta = meta.copy()

    # Only cycles that are actual discharge tests contribute to SoH
    discharge_idx = meta["type"] == "discharge"

    meta["capacity_est_Ah"] = np.nan
    meta["avg_temperature_C"] = np.nan
    meta["avg_voltage_V"] = np.nan
    meta["active_time_s"] = np.nan

    # --- Step 1: estimate capacity from raw BMS signals for every discharge file ---
    for idx in meta[discharge_idx].index:
        fname = meta.at[idx, "filename"]
        file_path = os.path.join(data_dir, fname)

        if not os.path.isfile(file_path):
            print(f"[WARN] Missing data file: {file_path}")
            continue

        df_cycle = pd.read_csv(file_path)

        stats = estimate_capacity_from_bms(df_cycle)
        meta.at[idx, "capacity_est_Ah"] = stats["capacity_Ah"]
        meta.at[idx, "avg_temperature_C"] = stats["avg_temperature_C"]
        meta.at[idx, "avg_voltage_V"] = stats["avg_voltage_V"]
        meta.at[idx, "active_time_s"] = stats["active_time_s"]

    # --- Step 2: decide reference capacity (SoH = capacity / reference_capacity) ---
    # We’ll do it PER BATTERY_ID so each cell is normalized to its own "fresh" capacity.
    meta["reference_capacity_Ah"] = np.nan
    meta["SoH"] = np.nan
    meta["SoH_percent"] = np.nan

    # Choose what capacity to use:
    #   - If nominal_capacity_Ah is given, use that for all batteries.
    #   - Else: for each battery_id, use the max estimated capacity over all discharge cycles.
    group_cols = ["battery_id"]
    for bid, group in meta[discharge_idx].groupby("battery_id"):
        if nominal_capacity_Ah is not None:
            ref_cap = float(nominal_capacity_Ah)
        else:
            # use maximum of estimated capacity for that battery
            ref_cap = float(group["capacity_est_Ah"].max())

        if ref_cap <= 0 or np.isnan(ref_cap):
            continue

        # SoH for all discharge cycles of this battery
        idxs = group.index
        meta.loc[idxs, "reference_capacity_Ah"] = ref_cap
        meta.loc[idxs, "SoH"] = meta.loc[idxs, "capacity_est_Ah"] / ref_cap
        meta.loc[idxs, "SoH_percent"] = meta.loc[idxs, "SoH"] * 100.0

    # Optional: you can also propagate SoH from nearest discharge to charge/impedance tests
    # by merging on test_id or uid, if you want SoH everywhere.

    # Save
    meta.to_csv(save_path, index=False)
    print(f"Saved SoH-annotated metadata to: {save_path}")

    return meta


In [12]:
# Adjust these paths as per your environment
metadata_path = "/kaggle/input/nasa-battery-dataset/cleaned_dataset/metadata.csv"
data_dir = "/kaggle/input/nasa-battery-dataset/cleaned_dataset/data"  # folder containing 00001.csv, 00002.csv, ...

# If you know the nominal capacity (e.g., 2.0 Ah), pass it here.
# If not, leave nominal_capacity_Ah=None and it will infer per battery.
meta_with_soh = compute_soh_for_all_cycles(
    metadata_path=metadata_path,
    data_dir=data_dir,
    nominal_capacity_Ah=None,          # or something like 2.0
    save_path="/kaggle/working/metadata_with_soh.csv"
)

# Check SoH for the first few discharge cycles
print(
    meta_with_soh[meta_with_soh["type"] == "discharge"][
        ["battery_id", "filename", "capacity_est_Ah", "reference_capacity_Ah", "SoH", "SoH_percent"]
    ].head()
)


Saved SoH-annotated metadata to: /kaggle/working/metadata_with_soh.csv
   battery_id   filename  capacity_est_Ah  reference_capacity_Ah       SoH  \
0       B0047  00001.csv         1.705850                1.70585  1.000000   
4       B0047  00005.csv         1.548554                1.70585  0.907790   
6       B0047  00007.csv         1.532332                1.70585  0.898281   
8       B0047  00009.csv         1.511640                1.70585  0.886150   
10      B0047  00011.csv         1.495247                1.70585  0.876541   

    SoH_percent  
0    100.000000  
4     90.779036  
6     89.828065  
8     88.615042  
10    87.654090  


In [13]:
df_234 = pd.read_csv("/kaggle/input/nasa-battery-dataset/cleaned_dataset/metadata.csv")

In [14]:
df_234

,type,start_time,ambient_temperature,battery_id,test_id,uid,filename,Capacity,Re,Rct
0,discharge,[2010. 7. 21. 15. 0. ...,4,B0047,0,1,00001.csv,1.6743047446975208,NaN,NaN
1,impedance,[2010. 7. 21. 16. 53. ...,24,B0047,1,2,00002.csv,NaN,0.05605783343888099,0.20097016584458333
2,charge,[2010. 7. 21. 17. 25. ...,4,B0047,2,3,00003.csv,NaN,NaN,NaN
3,impedance,[2010 7 21 20 31 5],24,B0047,3,4,00004.csv,NaN,0.05319185850921101,0.16473399914864734
4,discharge,[2.0100e+03 7.0000e+00 2.1000e+01 2.1000e+01 2...,4,B0047,4,5,00005.csv,1.5243662105099023,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
7560,impedance,[2010. 9. 30. 7. 36. ...,24,B0055,247,7561,07561.csv,NaN,0.0968087979207628,0.15489738203707232
7561,discharge,[2010. 9. 30. 8. 8. ...,4,B0055,248,7562,07562.csv,1.0201379996149256,NaN,NaN
7562,charge,[2010. 9. 30. 8. 48. 54.25],4,B0055,249,7563,07563.csv,NaN,NaN,NaN
7563,discharge,[2010. 9. 30. 11. 50. ...,4,B0055,250,7564,07564.csv,0.9907591663373165,NaN,NaN


In [15]:
import os
import numpy as np
import pandas as pd


def estimate_capacity_from_bms(
    df: pd.DataFrame,
    time_col: str = "Time",
    current_col: str = "Current_measured",
    load_col: str = "Current_load",
    current_noise_threshold: float = 1e-3,
    load_threshold: float = 0.5,
) -> float:
    """
    Estimate discharge capacity (Ah) from a single BMS file.

    Uses:
      - Time + Current_measured  -> coulomb counting
      - Current_load             -> only integrate when load is actually ON

    Assumptions:
      - Discharge current is NEGATIVE (Current_measured < 0 during discharge).
      - Current_load ~ 1.0 when actively discharging, ~0 when idle.

    Returns:
        capacity_Ah: estimated discharged capacity for this cycle (float)
    """
    d = df.copy()
    d = d.sort_values(time_col).reset_index(drop=True)

    # Time step in seconds
    d["delta_t"] = d[time_col].diff().fillna(0.0)

    # Clean current (remove tiny noise)
    I = d[current_col].astype(float).to_numpy()
    I_clean = np.where(np.abs(I) < current_noise_threshold, 0.0, I)
    d["I_clean"] = I_clean

    # Only count time when load is actually ON
    active = d[load_col] > load_threshold
    d["delta_t_active"] = np.where(active, d["delta_t"], 0.0)

    # Discharge is negative current -> take negative part and flip sign
    I_discharge = np.clip(d["I_clean"], None, 0.0)  # <= 0
    delta_Ah = -I_discharge * d["delta_t_active"] / 3600.0  # Ah

    capacity_Ah = float(delta_Ah.sum())
    return capacity_Ah


In [16]:
def fill_missing_capacity(
    metadata_path: str,
    data_dir: str,
    save_path: str = None,
    use_bms_for_discharge: bool = True,
) -> pd.DataFrame:
    """
    Fill missing values in the Capacity column of metadata:

    1) Optionally compute Capacity for missing DISCHARGE rows using BMS files.
    2) For each battery_id, interpolate Capacity over cycles (uid) and
       forward/backward fill to get a smooth Capacity_filled with no NaNs.

    Args:
        metadata_path: path to metadata.csv
        data_dir: folder where the BMS CSV files (e.g. 00001.csv, 00002.csv...) are stored
        save_path: optional path to save updated metadata
        use_bms_for_discharge: if True, compute Capacity from BMS for missing discharge rows

    Returns:
        metadata DataFrame with a new column:
            - Capacity_filled
    """
    meta = pd.read_csv(metadata_path)

    # Ensure Capacity is numeric (coerce strings / 'nan' to NaN)
    meta["Capacity"] = pd.to_numeric(meta["Capacity"], errors="coerce")

    # --- Step 1: for discharge rows with missing Capacity, compute from BMS ---
    if use_bms_for_discharge:
        mask = (meta["type"] == "discharge") & (meta["Capacity"].isna())

        for idx in meta[mask].index:
            fname = meta.at[idx, "filename"]
            fpath = os.path.join(data_dir, fname)

            if not os.path.isfile(fpath):
                print(f"[WARN] BMS file not found: {fpath}")
                continue

            df_cycle = pd.read_csv(fpath)
            cap_est = estimate_capacity_from_bms(df_cycle)
            meta.at[idx, "Capacity"] = cap_est

    # --- Step 2: per battery, interpolate & fill for all rows (charge + impedance too) ---
    # Sort so interpolation is along time/cycle order
    meta = meta.sort_values(["battery_id", "uid"]).reset_index(drop=True)

    # Create a fully filled capacity column
    meta["Capacity_filled"] = meta.groupby("battery_id")["Capacity"].transform(
        lambda s: s.interpolate().ffill().bfill()
    )

    # Optionally save
    if save_path is not None:
        meta.to_csv(save_path, index=False)
        print(f"Saved metadata with filled capacities to: {save_path}")

    return meta


In [18]:
metadata_path = "/kaggle/input/nasa-battery-dataset/cleaned_dataset/metadata.csv"     # or full path, e.g. "/mnt/data/metadata.csv"
data_dir = "/kaggle/input/nasa-battery-dataset/cleaned_dataset/data/"                     # folder that has 00001.csv, 00002.csv, ...

meta_with_cap = fill_missing_capacity(
    metadata_path=metadata_path,
    data_dir=data_dir,
    save_path="metadata_with_capacity_filled.csv",  # optional
    use_bms_for_discharge=True
)

# Quick check
print(
    meta_with_cap[["battery_id", "type", "filename", "Capacity", "Capacity_filled"]]
    .head(10)
)


Saved metadata with filled capacities to: metadata_with_capacity_filled.csv
  battery_id       type   filename  Capacity  Capacity_filled
0      B0005     charge  05121.csv       NaN         1.856487
1      B0005  discharge  05122.csv  1.856487         1.856487
2      B0005     charge  05123.csv       NaN         1.851407
3      B0005  discharge  05124.csv  1.846327         1.846327
4      B0005     charge  05125.csv       NaN         1.840838
5      B0005  discharge  05126.csv  1.835349         1.835349
6      B0005     charge  05127.csv       NaN         1.835306
7      B0005  discharge  05128.csv  1.835263         1.835263
8      B0005     charge  05129.csv       NaN         1.834954
9      B0005  discharge  05130.csv  1.834646         1.834646


/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


In [20]:
import os
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import cross_val_score, KFold


# ======================================================
# 1) BMS feature extraction from each filename (00001.csv, etc.)
# ======================================================

def compute_bms_features_for_file(file_path: str):
    """
    Compute engineered features from one cycle's BMS time series.

    Expects columns:
        Voltage_measured, Current_measured, Temperature_measured,
        Current_load, Voltage_load, Time

    Returns a dict with BMS-based features.
    If file does not exist or fails, returns NaNs for all features.
    """
    feature_names = [
        "bms_disc_capacity_ah",
        "bms_charge_capacity_ah",
        "bms_active_time_s",
        "bms_mean_temp",
        "bms_max_temp",
        "bms_mean_voltage",
        "bms_min_voltage",
        "bms_max_voltage",
        "bms_mean_current",
        "bms_max_current",
    ]

    if not os.path.isfile(file_path):
        return {name: np.nan for name in feature_names}

    try:
        df = pd.read_csv(file_path)
    except Exception:
        return {name: np.nan for name in feature_names}

    # Ensure required columns exist
    required_cols = [
        "Voltage_measured", "Current_measured", "Temperature_measured",
        "Current_load", "Voltage_load", "Time"
    ]
    if not all(col in df.columns for col in required_cols):
        return {name: np.nan for name in feature_names}

    df = df.sort_values("Time").reset_index(drop=True)
    df["delta_t"] = df["Time"].diff().fillna(0.0)

    # active = when load is actually applied
    load_threshold = 0.5
    active = df["Current_load"] > load_threshold
    df["delta_t_active"] = np.where(active, df["delta_t"], 0.0)

    I = df["Current_measured"].astype(float).values

    # Simple noise removal on current
    current_noise_threshold = 1e-3
    I_clean = np.where(np.abs(I) < current_noise_threshold, 0.0, I)

    # Separate discharge (<0) and charge (>0) currents
    I_discharge = np.clip(I_clean, None, 0.0)  # <= 0
    I_charge = np.clip(I_clean, 0.0, None)     # >= 0

    # Ah during active periods
    disc_Ah = (-I_discharge * df["delta_t_active"].values / 3600.0).sum()
    chg_Ah = (I_charge * df["delta_t_active"].values / 3600.0).sum()

    # Restrict stats to active time where dt_active > 0
    active_mask = (df["delta_t_active"] > 0)
    active_rows = df[active_mask] if active_mask.any() else df

    mean_temp = float(active_rows["Temperature_measured"].mean())
    max_temp = float(active_rows["Temperature_measured"].max())
    mean_voltage = float(active_rows["Voltage_measured"].mean())
    min_voltage = float(active_rows["Voltage_measured"].min())
    max_voltage = float(active_rows["Voltage_measured"].max())
    mean_current = float(active_rows["Current_measured"].mean())
    max_current = float(active_rows["Current_measured"].max())
    active_time_s = float(df["delta_t_active"].sum())

    return {
        "bms_disc_capacity_ah": disc_Ah,
        "bms_charge_capacity_ah": chg_Ah,
        "bms_active_time_s": active_time_s,
        "bms_mean_temp": mean_temp,
        "bms_max_temp": max_temp,
        "bms_mean_voltage": mean_voltage,
        "bms_min_voltage": min_voltage,
        "bms_max_voltage": max_voltage,
        "bms_mean_current": mean_current,
        "bms_max_current": max_current,
    }


def add_bms_features_to_metadata(metadata: pd.DataFrame, data_dir: str) -> pd.DataFrame:
    """
    For each row in metadata, read its 'filename' CSV from data_dir
    and add BMS-based features as new columns.
    """
    bms_feature_names = [
        "bms_disc_capacity_ah",
        "bms_charge_capacity_ah",
        "bms_active_time_s",
        "bms_mean_temp",
        "bms_max_temp",
        "bms_mean_voltage",
        "bms_min_voltage",
        "bms_max_voltage",
        "bms_mean_current",
        "bms_max_current",
    ]

    for name in bms_feature_names:
        if name not in metadata.columns:
            metadata[name] = np.nan

    for idx, row in metadata.iterrows():
        fname = row.get("filename", None)
        if not isinstance(fname, str):
            continue

        fpath = os.path.join(data_dir, fname)
        feats = compute_bms_features_for_file(fpath)

        for k, v in feats.items():
            metadata.at[idx, k] = v

    return metadata


# ======================================================
# 2) Generic function to train + fill one target (Capacity / Re / Rct)
# ======================================================

def train_and_fill_target(
    metadata: pd.DataFrame,
    target_col: str,
    feature_cols: list,
    categorical_cols: list,
    random_state: int = 42,
):
    """
    Train a regression model for `target_col` using rows where it's present,
    then predict missing values.

    - Uses HistGradientBoostingRegressor (tree-based, strong baseline)
    - Handles numeric + categorical features
    - Imputes missing features internally
    - Prints basic cross-val scores (MAE, R^2)

    Returns:
        metadata with a new column:
            f"{target_col}_pred"
        and the trained model.
    """
    df = metadata.copy()

    # Ensure target is numeric
    df[target_col] = pd.to_numeric(df[target_col], errors="coerce")

    # Build feature matrix
    X = df[feature_cols].copy()
    y = df[target_col]

    # Train only on rows where target is known
    mask_train = y.notna()
    X_train = X[mask_train]
    y_train = y[mask_train]

    if X_train.empty:
        print(f"[WARN] No training data available for target '{target_col}'.")
        df[f"{target_col}_pred"] = df[target_col]
        return df, None

    # Identify numeric features as the rest of feature_cols
    numeric_cols = [c for c in feature_cols if c not in categorical_cols]

    # Preprocessing
    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )
    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_cols),
            ("cat", categorical_transformer, categorical_cols),
        ]
    )

    # Regressor – gradient boosting (strong non-linear model)
    regressor = HistGradientBoostingRegressor(
        random_state=random_state,
        max_depth=None,
        max_iter=300,
        learning_rate=0.05,
    )

    model = Pipeline(
        steps=[
            ("preprocess", preprocessor),
            ("regressor", regressor),
        ]
    )

    # Cross-validation to get a sense of accuracy on known data
    cv = KFold(n_splits=5, shuffle=True, random_state=random_state)

    mae_scores = -cross_val_score(
        model, X_train, y_train, cv=cv, scoring="neg_mean_absolute_error"
    )
    r2_scores = cross_val_score(
        model, X_train, y_train, cv=cv, scoring="r2"
    )

    print(f"Target: {target_col}")
    print(f"  MAE (5-fold CV): {mae_scores.mean():.4f} ± {mae_scores.std():.4f}")
    print(f"  R²  (5-fold CV): {r2_scores.mean():.4f} ± {r2_scores.std():.4f}")

    # Fit on all available labeled data
    model.fit(X_train, y_train)

    # Predict for rows where target is missing
    mask_missing = y.isna()
    if mask_missing.any():
        X_missing = X[mask_missing]
        y_pred = model.predict(X_missing)
        df[f"{target_col}_pred"] = df[target_col]
        df.loc[mask_missing, f"{target_col}_pred"] = y_pred
    else:
        # No missing, just copy
        df[f"{target_col}_pred"] = df[target_col]

    return df, model


# ======================================================
# 3) End-to-end usage
# ======================================================

if __name__ == "__main__":
    # ---- Adjust these paths ----
    METADATA_PATH = "/kaggle/input/nasa-battery-dataset/cleaned_dataset/metadata.csv"  # path to your metadata
    DATA_DIR = "/kaggle/input/nasa-battery-dataset/cleaned_dataset/data"                    # folder containing 00001.csv, 00002.csv, ...
    SAVE_PATH = "/kaggle/working/metadata_with_predictions.csv"

    # 1) Load metadata
    meta = pd.read_csv(METADATA_PATH)

    # 2) Add BMS-based features from each filename CSV
    meta = add_bms_features_to_metadata(meta, DATA_DIR)

    # 3) Define features to use for learning
    #    We avoid using target columns themselves (Capacity, Re, Rct) as features
    #    to prevent leakage.
    feature_cols = [
        "type",
        "ambient_temperature",
        "battery_id",
        "test_id",
        "uid",
        # BMS engineered features:
        "bms_disc_capacity_ah",
        "bms_charge_capacity_ah",
        "bms_active_time_s",
        "bms_mean_temp",
        "bms_max_temp",
        "bms_mean_voltage",
        "bms_min_voltage",
        "bms_max_voltage",
        "bms_mean_current",
        "bms_max_current",
    ]

    # Some rows may not have BMS features -> fine, we impute later
    # Categorical features among the feature_cols
    categorical_cols = ["type", "battery_id"]

    # 4) Train + fill CAPACITY
    meta, model_capacity = train_and_fill_target(
        metadata=meta,
        target_col="Capacity",
        feature_cols=feature_cols,
        categorical_cols=categorical_cols,
        random_state=42,
    )

    # 5) Train + fill Re
    meta, model_re = train_and_fill_target(
        metadata=meta,
        target_col="Re",
        feature_cols=feature_cols,
        categorical_cols=categorical_cols,
        random_state=42,
    )

    # 6) Train + fill Rct
    meta, model_rct = train_and_fill_target(
        metadata=meta,
        target_col="Rct",
        feature_cols=feature_cols,
        categorical_cols=categorical_cols,
        random_state=42,
    )

    # 7) Save final metadata with predicted values
    meta.to_csv(SAVE_PATH, index=False)
    print(f"\nSaved metadata with predicted Capacity/Re/Rct to: {SAVE_PATH}")

    # Quick look at the result
    cols_to_show = [
        "battery_id", "type", "filename",
        "Capacity", "Capacity_pred",
        "Re", "Re_pred",
        "Rct", "Rct_pred",
    ]
    print(meta[cols_to_show].head(20))


Target: Capacity
  MAE (5-fold CV): 0.0146 ± 0.0016
  R²  (5-fold CV): 0.9920 ± 0.0034


ValueError: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/sklearn/model_selection/_validation.py", line 686, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/usr/local/lib/python3.11/dist-packages/sklearn/pipeline.py", line 405, in fit
    self._final_estimator.fit(Xt, y, **fit_params_last_step)
  File "/usr/local/lib/python3.11/dist-packages/sklearn/ensemble/_hist_gradient_boosting/gradient_boosting.py", line 361, in fit
    X, y = self._validate_data(X, y, dtype=[X_DTYPE], force_all_finite=False)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/s

In [21]:
import os
import numpy as np
import pandas as pd
import sklearn

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import cross_val_score, KFold


# ======================================================
# 1) BMS feature extraction from each filename (00001.csv, etc.)
# ======================================================

def compute_bms_features_for_file(file_path: str):
    """
    Compute engineered features from one cycle's BMS time series.

    Expects columns in the CSV:
        Voltage_measured, Current_measured, Temperature_measured,
        Current_load, Voltage_load, Time

    Returns a dict with BMS-based features.
    If file does not exist or fails, returns NaNs for all features.
    """
    feature_names = [
        "bms_disc_capacity_ah",
        "bms_charge_capacity_ah",
        "bms_active_time_s",
        "bms_mean_temp",
        "bms_max_temp",
        "bms_mean_voltage",
        "bms_min_voltage",
        "bms_max_voltage",
        "bms_mean_current",
        "bms_max_current",
    ]

    if not os.path.isfile(file_path):
        return {name: np.nan for name in feature_names}

    try:
        df = pd.read_csv(file_path)
    except Exception:
        return {name: np.nan for name in feature_names}

    required_cols = [
        "Voltage_measured", "Current_measured", "Temperature_measured",
        "Current_load", "Voltage_load", "Time"
    ]
    if not all(col in df.columns for col in required_cols):
        return {name: np.nan for name in feature_names}

    df = df.sort_values("Time").reset_index(drop=True)
    df["delta_t"] = df["Time"].diff().fillna(0.0)

    # active = when load is actually applied
    load_threshold = 0.5
    active = df["Current_load"] > load_threshold
    df["delta_t_active"] = np.where(active, df["delta_t"], 0.0)

    # Current cleaning
    I = df["Current_measured"].astype(float).values
    current_noise_threshold = 1e-3
    I_clean = np.where(np.abs(I) < current_noise_threshold, 0.0, I)

    # Separate discharge (<0) and charge (>0) currents
    I_discharge = np.clip(I_clean, None, 0.0)  # <= 0
    I_charge = np.clip(I_clean, 0.0, None)     # >= 0

    # Ah during active periods
    disc_Ah = (-I_discharge * df["delta_t_active"].values / 3600.0).sum()
    chg_Ah = (I_charge * df["delta_t_active"].values / 3600.0).sum()

    # Restrict stats to active time where dt_active > 0
    active_mask = (df["delta_t_active"] > 0)
    active_rows = df[active_mask] if active_mask.any() else df

    mean_temp = float(active_rows["Temperature_measured"].mean())
    max_temp = float(active_rows["Temperature_measured"].max())
    mean_voltage = float(active_rows["Voltage_measured"].mean())
    min_voltage = float(active_rows["Voltage_measured"].min())
    max_voltage = float(active_rows["Voltage_measured"].max())
    mean_current = float(active_rows["Current_measured"].mean())
    max_current = float(active_rows["Current_measured"].max())
    active_time_s = float(df["delta_t_active"].sum())

    return {
        "bms_disc_capacity_ah": disc_Ah,
        "bms_charge_capacity_ah": chg_Ah,
        "bms_active_time_s": active_time_s,
        "bms_mean_temp": mean_temp,
        "bms_max_temp": max_temp,
        "bms_mean_voltage": mean_voltage,
        "bms_min_voltage": min_voltage,
        "bms_max_voltage": max_voltage,
        "bms_mean_current": mean_current,
        "bms_max_current": max_current,
    }


def add_bms_features_to_metadata(metadata: pd.DataFrame, data_dir: str) -> pd.DataFrame:
    """
    For each row in metadata, read its 'filename' CSV from data_dir
    and add BMS-based features as new columns.
    """
    bms_feature_names = [
        "bms_disc_capacity_ah",
        "bms_charge_capacity_ah",
        "bms_active_time_s",
        "bms_mean_temp",
        "bms_max_temp",
        "bms_mean_voltage",
        "bms_min_voltage",
        "bms_max_voltage",
        "bms_mean_current",
        "bms_max_current",
    ]

    for name in bms_feature_names:
        if name not in metadata.columns:
            metadata[name] = np.nan

    for idx, row in metadata.iterrows():
        fname = row.get("filename", None)
        if not isinstance(fname, str):
            continue

        fpath = os.path.join(data_dir, fname)
        feats = compute_bms_features_for_file(fpath)

        for k, v in feats.items():
            metadata.at[idx, k] = v

    return metadata


# ======================================================
# 2) Training + filling a single target (Capacity / Re / Rct)
# ======================================================

def train_and_fill_target(
    metadata: pd.DataFrame,
    target_col: str,
    feature_cols: list,
    categorical_cols: list,
    random_state: int = 42,
):
    """
    Train a regression model for `target_col` using rows where it's present,
    then predict missing values.

    - Uses HistGradientBoostingRegressor (tree-based, strong baseline)
    - Handles numeric + categorical features
    - Imputes missing features internally
    - Prints basic cross-val scores (MAE, R^2)

    Returns:
        metadata with a new column:
            f"{target_col}_pred"
        and the trained model.
    """
    df = metadata.copy()

    # Ensure target is numeric
    df[target_col] = pd.to_numeric(df[target_col], errors="coerce")

    # Feature matrix
    X = df[feature_cols].copy()
    y = df[target_col]

    # Train only on rows where target is known
    mask_train = y.notna()
    X_train = X[mask_train]
    y_train = y[mask_train]

    if X_train.empty:
        print(f"[WARN] No training data available for target '{target_col}'.")
        df[f"{target_col}_pred"] = df[target_col]
        return df, None

    # Numeric features = rest of feature_cols
    numeric_cols = [c for c in feature_cols if c not in categorical_cols]

    # --- Preprocessing: force DENSE output so HGBR is happy ---
    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    # Handle sklearn version difference for OneHotEncoder dense output
    ohe_kwargs = {"handle_unknown": "ignore"}
    ver = tuple(int(x) for x in sklearn.__version__.split(".")[:2])
    if ver >= (1, 2):
        ohe_kwargs["sparse_output"] = False  # sklearn >= 1.2
    else:
        ohe_kwargs["sparse"] = False         # sklearn < 1.2

    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(**ohe_kwargs)),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_cols),
            ("cat", categorical_transformer, categorical_cols),
        ],
        sparse_threshold=0.0,  # force dense
    )

    # Regressor – gradient boosting (needs dense)
    regressor = HistGradientBoostingRegressor(
        random_state=random_state,
        max_depth=None,
        max_iter=300,
        learning_rate=0.05,
    )

    model = Pipeline(
        steps=[
            ("preprocess", preprocessor),
            ("regressor", regressor),
        ]
    )

    # --- Cross-validation (guard for small sample sizes) ---
    n_train = len(X_train)
    n_splits = min(5, n_train)

    if n_splits > 1:
        cv = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)

        mae_scores = -cross_val_score(
            model, X_train, y_train, cv=cv, scoring="neg_mean_absolute_error"
        )
        r2_scores = cross_val_score(
            model, X_train, y_train, cv=cv, scoring="r2"
        )

        print(f"Target: {target_col}")
        print(f"  MAE ({n_splits}-fold CV): {mae_scores.mean():.4f} ± {mae_scores.std():.4f}")
        print(f"  R²  ({n_splits}-fold CV): {r2_scores.mean():.4f} ± {r2_scores.std():.4f}")
    else:
        print(f"Target: {target_col} – only {n_train} labeled samples, skipping CV.")

    # Fit on all labeled data
    model.fit(X_train, y_train)

    # Predict for missing targets
    mask_missing = y.isna()
    df[f"{target_col}_pred"] = df[target_col]

    if mask_missing.any():
        X_missing = X[mask_missing]
        y_pred = model.predict(X_missing)
        df.loc[mask_missing, f"{target_col}_pred"] = y_pred

    return df, model


# ======================================================
# 3) End-to-end usage
# ======================================================

if __name__ == "__main__":
    # ---- Adjust these paths as needed ----
    METADATA_PATH = "/kaggle/input/nasa-battery-dataset/cleaned_dataset/metadata.csv"  # path to your metadata
    DATA_DIR = "/kaggle/input/nasa-battery-dataset/cleaned_dataset/data"                    # folder containing 00001.csv, 00002.csv, ...
    SAVE_PATH = "/kaggle/working/metadata_with_predictions.csv"

    # 1) Load metadata (columns: type, start_time, ambient_temperature,
    #                   battery_id, test_id, uid, filename, Capacity, Re, Rct)
    meta = pd.read_csv(METADATA_PATH)

    # 2) Add BMS-based features from each filename CSV
    meta = add_bms_features_to_metadata(meta, DATA_DIR)

    # 3) Define feature columns for learning
    #    (avoid target columns themselves to prevent leakage)
    feature_cols = [
        "type",
        "ambient_temperature",
        "battery_id",
        "test_id",
        "uid",
        # BMS engineered features:
        "bms_disc_capacity_ah",
        "bms_charge_capacity_ah",
        "bms_active_time_s",
        "bms_mean_temp",
        "bms_max_temp",
        "bms_mean_voltage",
        "bms_min_voltage",
        "bms_max_voltage",
        "bms_mean_current",
        "bms_max_current",
    ]

    # Categorical features among the above
    categorical_cols = ["type", "battery_id"]

    # 4) Train + fill Capacity
    meta, model_capacity = train_and_fill_target(
        metadata=meta,
        target_col="Capacity",
        feature_cols=feature_cols,
        categorical_cols=categorical_cols,
        random_state=42,
    )

    # 5) Train + fill Re
    meta, model_re = train_and_fill_target(
        metadata=meta,
        target_col="Re",
        feature_cols=feature_cols,
        categorical_cols=categorical_cols,
        random_state=42,
    )

    # 6) Train + fill Rct
    meta, model_rct = train_and_fill_target(
        metadata=meta,
        target_col="Rct",
        feature_cols=feature_cols,
        categorical_cols=categorical_cols,
        random_state=42,
    )

    # 7) Save final metadata with predicted values
    meta.to_csv(SAVE_PATH, index=False)
    print(f"\nSaved metadata with predicted Capacity/Re/Rct to: {SAVE_PATH}")

    # Quick sanity check
    cols_to_show = [
        "battery_id", "type", "filename",
        "Capacity", "Capacity_pred",
        "Re", "Re_pred",
        "Rct", "Rct_pred",
    ]
    print(meta[cols_to_show].head(20))


Target: Capacity
  MAE (5-fold CV): 0.0146 ± 0.0016
  R²  (5-fold CV): 0.9920 ± 0.0034
Target: Re
  MAE (5-fold CV): 1416005832248.8608 ± 624712297807.5349
  R²  (5-fold CV): -69180456055299268608.0000 ± 99070618806384263168.0000
Target: Rct
  MAE (5-fold CV): 3004451167705.5518 ± 1325501312981.0132
  R²  (5-fold CV): -1858449081849222004736.0000 ± 3261231583353939427328.0000

Saved metadata with predicted Capacity/Re/Rct to: /kaggle/working/metadata_with_predictions.csv
   battery_id       type   filename  Capacity  Capacity_pred        Re  \
0       B0047  discharge  00001.csv  1.674305       1.674305       NaN   
1       B0047  impedance  00002.csv       NaN       1.105900  0.056058   
2       B0047     charge  00003.csv       NaN       1.402077       NaN   
3       B0047  impedance  00004.csv       NaN       1.107108  0.053192   
4       B0047  discharge  00005.csv  1.524366       1.524366       NaN   
5       B0047     charge  00006.csv       NaN       1.401114       NaN   
6     

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
